# ML-04: Search Intelligence Data Contract

**Lane:** Content-opportunity scoring — expected CTR regression on
`fact_content_daily_performance`, mid-panel month `month=2026-03`.

**Rule followed:** core reasoning first, AI used only to stress-test and verify against real
data — not to author the contract. Every number below was run against the real warehouse
(`FlyRank/internship-warehouse` on HuggingFace, accessed via a Colab session with an HF token),
not assumed or estimated.


## Part 1 — The Contract, in Plain Words

1. **What one row means:** one (client, content item) pair, aggregated from daily rows in
   `fact_content_daily_performance` where `report_date` falls in March 2026 — sum of clicks,
   sum of impressions, and an **impression-weighted** average position (not a simple mean,
   so a low-traffic day doesn't drag the average as much as a high-traffic day).

2. **Which table(s):** `fact_content_daily_performance` (`month=2026-03` partition) as the
   primary source, joined with `dim_content` for `main_intent`, `search_volume`, and
   `content_visible_query_count`, and filtered via `dim_clients` to clients with both
   `has_gsc_access` and `has_ga4_access` true.

3. **Which time window:** the calendar month `2026-03` (March 1–31, 2026) — a mid-panel month,
   not the sealed `fact_content_daily_performance_sample.parquet` / `fact_content_query_90d`
   snapshot, which is reserved as a sealed test month (it turned out to be the final month,
   June 2026, confirmed by inspecting `window_start`/`window_end`).

4. **What I'd predict or rank — label or proxy:** predict expected CTR
   (`clicks_march / impressions_march`) via regression; the opportunity score is the residual
   (`actual_CTR - predicted_CTR`). This is a **direct target**, computed from the data itself —
   not a proxy or a rule-derived label.

5. **One thing deliberately excluded, and why:** clients without full `gsc_and_ga4` access.
   Incomplete GSC or GA4 data would corrupt the position/CTR features for those clients —
   confirmed below (Query 3) that this exclusion still keeps ~78% of March's rows, so it's not
   gutting the dataset.


## Part 2 — Prove Three Facts with Three Small Queries

Run against `month=2026-03` via DuckDB's `hf://` reader, authenticated with an HF token
(`CREATE SECRET ... TYPE huggingface`). Requires `httpfs` loaded and a valid `HF_TOKEN` with
access granted to the gated `FlyRank/internship-warehouse` dataset.


In [ ]:
import duckdb
import os

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # set this in your own environment / Colab secret
if HF_TOKEN:
    con.sql(f"""
        CREATE SECRET hf_secret (
            TYPE huggingface,
            TOKEN '{HF_TOKEN}'
        );
    """)
else:
    print("No HF_TOKEN found in this environment — queries below will fail until one is set.")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
CLIENTS_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"


In [ ]:
# Query 1: Grain check — is report_date x client x content really one row?
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM read_parquet('{MARCH_PATH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").show()


**Proven result (run 2026, via Colab with HF auth): `0 rows` returned.**
Grain claim confirmed — `report_date x client_hash_id x content_hash_id` is a true one-row grain,
no duplicates in March.


In [ ]:
# Query 2: Row count and date span for the slice
q2 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date
    FROM read_parquet('{MARCH_PATH}')
""").show()


**Proven result: `total_rows = 9,841,378`, `earliest_date = 2026-03-01`, `latest_date = 2026-03-31`.**
Full calendar month, no gaps at either end.


In [ ]:
# Query 3: Availability after filtering to clients with IS TRUE gsc_and_ga4 access
q3 = con.sql(f"""
    SELECT COUNT(*) AS surviving_rows
    FROM read_parquet('{MARCH_PATH}') f
    JOIN read_parquet('{CLIENTS_PATH}') c
      ON f.client_hash_id = c.client_hash_id
    WHERE c.has_gsc_access IS TRUE
      AND c.has_ga4_access IS TRUE
""").show()


**Proven result: `surviving_rows = 7,700,646`** — about **78%** of the full month's rows survive
the `gsc_and_ga4`-only filter (7,700,646 / 9,841,378). The exclusion in Question 5 is real but not
destructive to the dataset.


## Part 3 — Five Features, Max

Built from the same March 2026 slice. Each feature includes a one-line
"knowable at the decision moment because…" justification.

| Feature | Knowable at decision moment because… |
|---|---|
| `avg_position_march` (impression-weighted) | it's simply where the page currently ranks — observable the moment you'd score the item, no future data needed |
| `impressions_march` (summed) | it's search visibility the item already received — a completed, observed fact by month's end |
| `main_intent` (from `dim_content`) | it's a property of the content itself, set at creation — doesn't depend on any future performance outcome |
| `search_volume` (from `dim_content`) | it's a keyword-market fact, independent of how this specific content item performs |
| `content_visible_query_count` | it reflects the item's current query reach — a structural fact, not a downstream outcome |

**Deliberately left out for now:** `backlinks` (possible causal loop — backlinks could increase
*because* content gets promoted after a high opportunity score, which would make it a
feedback-contaminated feature rather than a clean predictor) and `last_optimized_date` /
`optimization_eligible_date` (reserved for the Part 4 trap — these are the ones that sit closest
to leaking the outcome).


In [ ]:
# Build the March feature frame: aggregate daily -> one row per (client, content item),
# then join dim_content for main_intent / search_volume / content_visible_query_count.
feature_query = f"""
    WITH march_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(clicks)      AS clicks_march,
            SUM(impressions) AS impressions_march,
            SUM(avg_position * impressions) / NULLIF(SUM(impressions), 0) AS avg_position_march
        FROM read_parquet('{MARCH_PATH}')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.clicks_march,
        m.impressions_march,
        m.avg_position_march,
        d.main_intent,
        d.search_volume,
        d.content_visible_query_count,
        m.clicks_march * 1.0 / NULLIF(m.impressions_march, 0) AS actual_ctr_march
    FROM march_agg m
    JOIN read_parquet('{CONTENT_PATH}') d
      ON m.content_hash_id = d.content_hash_id
    WHERE m.impressions_march > 0
"""

df = con.sql(feature_query).df()
df.head()


## Part 4 — The Trap

**Deliberate leak:** `ctr_tier`, a categorical column built by splitting `actual_ctr_march` into
terciles (low / medium / high). This isn't a real predictive feature — it's a rounded restatement
of the label itself. Including it should make the model's error look artificially small, not
because it learned anything about position, intent, or search volume, but because it can just
map each tier back to roughly that tier's average CTR.

**Prediction before running:** MAE will drop sharply with `ctr_tier` included, because the model
no longer needs to infer CTR from the five legitimate features — it can shortcut straight to the
tier's average. That drop says nothing about whether `avg_position_march`, `main_intent`, etc.
actually predict CTR well; it only shows a model can recover a number when handed a rounded
version of that same number. The honest MAE is only the one measured **without** `ctr_tier`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

model_df = df.dropna(subset=["actual_ctr_march", "avg_position_march", "main_intent"]).copy()

# Deliberate leak: tier built directly from the label
model_df["ctr_tier"] = pd.qcut(model_df["actual_ctr_march"], q=3, labels=["low", "medium", "high"])

numeric_features = ["avg_position_march", "impressions_march", "search_volume", "content_visible_query_count"]
categorical_features_clean = ["main_intent"]
categorical_features_leaky = ["main_intent", "ctr_tier"]

def run_model(cat_features):
    pre = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ])
    pipe = Pipeline([("pre", pre), ("reg", LinearRegression())])

    X = model_df[numeric_features + cat_features]
    y = model_df["actual_ctr_march"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    return mean_absolute_error(y_test, preds)

mae_leaky = run_model(categorical_features_leaky)
mae_honest = run_model(categorical_features_clean)

print(f"MAE WITH ctr_tier (leaked):    {mae_leaky:.5f}")
print(f"MAE WITHOUT ctr_tier (honest): {mae_honest:.5f}")
print(f"\nApparent improvement from the leak: {mae_honest - mae_leaky:.5f} "
      f"({(1 - mae_leaky / mae_honest) * 100:.1f}% smaller error)")


**The honest number is `mae_honest`, not `mae_leaky`.** The leaked version's improvement isn't
evidence the model got better at understanding content opportunity — `ctr_tier` is a near-copy of
the answer, so the model is just decoding a rounded version of the label instead of learning from
position, intent, impressions, or query count. Removing it and reporting `mae_honest` is the only
number that says anything real about whether the five legitimate features actually predict CTR.


## Self-Check

- **Can I state the contract in one breath?** Regression predicting March 2026 expected CTR for
  a (client, content item) pair, from position/impressions/intent/search-volume/query-count,
  scored by MAE against the directly-computed actual CTR — evaluated only on clients with full
  GSC+GA4 access.

- **Do I have real numbers, not descriptions, for the three proof queries?** Yes: 0 duplicate
  grain rows, 9,841,378 total March rows spanning 03-01 to 03-31, and 7,700,646 rows surviving
  the access filter (~78%).

- **Could I explain the trap live, without the notebook in front of me?** Yes: `ctr_tier` is a
  binned copy of the label. Including it produces a fake-good MAE because the model can shortcut
  straight to a tier average instead of learning from real predictors. The honest MAE — the one
  that matters — is the version without it.

- **Anything here I'm not fully sure I could defend?** `backlinks` was left out over a suspected
  feedback loop (content promoted after a high opportunity score might gain backlinks as a
  result) — I haven't actually verified this happens in the data; it's a reasoned exclusion, not
  a confirmed one. Worth checking before claiming it definitively if asked.
